In [ ]:
# LoRA를 사용하기 위한 Hugging Face의 PEFT 라이브러리 설치
!pip install -q peft

In [ ]:
print('Checking and upgrading torchao to resolve version incompatibility...')
!pip install --upgrade torchao
print('torchao upgrade complete. Please re-run the previous cells if the issue persists.')

Checking and upgrading torchao to resolve version incompatibility...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
torchao upgrade complete. Please re-run the previous cells if the issue persists.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DistilBertModel, AutoTokenizer, CLIPVisionModelWithProjection, CLIPProcessor
from peft import LoraConfig, get_peft_model
import pandas as pd

# 1. 환경 설정 및 하이퍼파라미터
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STAGE1_PATH = '/content/drive/MyDrive/checkpoints/stage1/best.pt'
LR = 1e-5  # Stage 2는 매우 작은 학습률로 조심스럽게!
LAMBDA_1, LAMBDA_2 = 1.0, 1.0 # Loss 가중치 (수식의 람다)

# 2. 모델 로드 및 LoRA 설정
TEXT_MODEL_NAME = 'sentence-transformers/clip-ViT-B-32-multilingual-v1'
VISION_MODEL_NAME = 'openai/clip-vit-base-patch32'

text_encoder = DistilBertModel.from_pretrained(TEXT_MODEL_NAME)
vision_encoder = CLIPVisionModelWithProjection.from_pretrained(VISION_MODEL_NAME)

# --- LoRA 적용 ---
text_lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_lin", "v_lin"], lora_dropout=0.1)
text_encoder = get_peft_model(text_encoder, text_lora_config)

vision_lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.1)
vision_encoder = get_peft_model(vision_encoder, vision_lora_config)

# 3. Head 정의 (Stage 1과 동일한 구조여야 함)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 256)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

# Redefine SceneHead to precisely match checkpoint['cls_head']'s inferred structure based on error messages
class SceneHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256),  # Corrected: input 512, output 256, matches checkpoint net.0.weight [256, 512]
            nn.ReLU(),            # Assumed position
            nn.Dropout(0.1),      # Assumed position for net.2
            nn.Linear(256, 13)    # Corrected: input 256, output 13, matches checkpoint net.3.weight [13, 256]
        )
    def forward(self, x):
        return self.net(x)

txt_proj_layer = nn.Linear(768, 512)
proj_head = ProjectionHead()
scene_head = SceneHead() # Instantiate the new SceneHead class

# 4. Stage 1 가중치 불러오기 (매우 중요!)
checkpoint = torch.load(STAGE1_PATH, map_location=device)
txt_proj_layer.load_state_dict(checkpoint['txt_proj_layer'])
proj_head.load_state_dict(checkpoint['proj_head'])
scene_head.load_state_dict(checkpoint['cls_head'])

# 모든 모델을 장치로 이동
text_encoder.to(device); vision_encoder.to(device)
txt_proj_layer.to(device); proj_head.to(device); scene_head.to(device)

# 5. 최적화 대상 설정 (LoRA 파라미터 + 모든 Head)
params = list(text_encoder.parameters()) + list(vision_encoder.parameters()) + \
         list(txt_proj_layer.parameters()) + list(proj_head.parameters()) + \
         list(scene_head.parameters())
optimizer = torch.optim.AdamW(params, lr=LR)

# 6. 학습 루프 예시 (Loss 수식 반영)
def train_step(images, input_ids, masks, labels):
    optimizer.zero_grad()

    # 특징 추출
    img_feat = vision_encoder(pixel_values=images).image_embeds # 512
    txt_out = text_encoder(input_ids=input_ids, attention_mask=masks)[0][:, 0, :] # 768
    txt_feat = txt_proj_layer(txt_out) # 512

    # 1) L_CLIP: 대조 학습 (256차원 투영 후 계산)
    i_emb = proj_head(img_feat)
    t_emb = proj_head(txt_feat)
    logits = i_emb @ t_emb.T * torch.exp(torch.tensor(0.07).to(device))
    targets = torch.arange(images.size(0)).to(device)
    loss_clip = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2

    # 2) L_CE: 장소 분류 (Classification)
    scene_logits = scene_head(img_feat)
    loss_ce = F.cross_entropy(scene_logits, labels)

    # 3) L_SupCon: 지도 대조 학습 (간략화된 버전)
    # 실제로는 같은 라벨끼리 뭉치게 하는 복잡한 로직이 들어가지만
    # 여기서는 구조적 합산만 표시합니다.
    loss_supcon = 0.0 # 필요 시 구현체 추가

    # 최종 통합 Loss
    total_loss = loss_clip + (LAMBDA_1 * loss_supcon) + (LAMBDA_2 * loss_ce)

    total_loss.backward()
    optimizer.step()
    return total_loss.item()

print("🚀 Stage 2: LoRA Fine-tuning 준비 완료!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.

🚀 Stage 2: LoRA Fine-tuning 준비 완료!


In [ ]:
# Cell 3. 설정값
# ====================================================
# 경로
CSV_PATH   = '/content/drive/MyDrive/metadata_stage1_ready.xlsx'
IMAGE_ROOT = '/content/drive/MyDrive/캡스톤 디자인 - Team Oasis/images_final'
SAVE_PATH  = '/content/drive/MyDrive/checkpoints/stage1/'
# ====================================================

# 차원
IMG_DIM  = 512   # CLIP 이미지 인코더 출력
TXT_DIM  = 768   # DistilBERT 출력
PROJ_DIM = 256   # ProjectionHead 출력

# scene 라벨 (13개)
SCENE_CLASSES = ['바다','산','숲','도시','호수','거리','전통','공원','전망','조형물','카페','온천','전시']
SCENE2IDX     = {s: i for i, s in enumerate(SCENE_CLASSES)}
NUM_CLASSES   = len(SCENE_CLASSES)

# 하이퍼파라미터
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR         = 1e-3
LAMBDA_CE  = 0.7

print(f'NUM_CLASSES: {NUM_CLASSES}')
print(f'IMG_DIM: {IMG_DIM}, TXT_DIM: {TXT_DIM}, PROJ_DIM: {PROJ_DIM}')

NUM_CLASSES: 13
IMG_DIM: 512, TXT_DIM: 768, PROJ_DIM: 256


In [ ]:
processor      = CLIPProcessor.from_pretrained(VISION_MODEL_NAME)

TEXT_MODEL_NAME   = 'sentence-transformers/clip-ViT-B-32-multilingual-v1'

text_encoder   = DistilBertModel.from_pretrained(TEXT_MODEL_NAME).to(device)
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/371 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
import os
import unicodedata
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# --- [추가] 한글 인코딩 문제 해결을 위한 정규화 함수 ---
def dn(text):
    if pd.isna(text): return ""
    return unicodedata.normalize('NFC', str(text)).strip()

# --- [추가] 실제 폴더 안의 파일들을 미리 스캔 (딱 한 번만 실행) ---
# 엑셀의 이름과 실제 파일명이 인코딩 차이로 다를 때를 대비해 매핑 테이블을 만듭니다.
print("🔎 이미지 폴더 스캔 중...")
actual_file_map = {dn(f): f for f in os.listdir(IMAGE_ROOT) if not f.startswith('.')}
print(f"✅ 총 {len(actual_file_map)}개의 실제 파일 인식 완료.")

class TravelDataset(Dataset):
    def __init__(self, df, image_root, processor, tokenizer, scene2idx):
        self.df         = df.reset_index(drop=True)
        self.image_root = image_root
        self.processor  = processor
        self.tokenizer  = tokenizer
        self.scene2idx  = scene2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. 엑셀에 적힌 이미지 아이디 정규화
        target_id = dn(row['image_id'])

        # 2. 매핑 테이블에서 실제 파일명 찾기
        real_file_name = actual_file_map.get(target_id)

        if real_file_name is None:
            # 파일이 없을 경우 에러 메시지를 명확히 출력
            raise FileNotFoundError(f"❌ 파일 없음: {target_id} (엑셀 행 번호: {idx})")

        img_path = os.path.join(self.image_root, real_file_name)

        # 이미지 로드 및 처리
        try:
            image = Image.open(img_path).convert('RGB')
            pixel_values = self.processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        except Exception as e:
            print(f"❌ 이미지 로드 실패: {img_path}")
            raise e

        # 텍스트 처리
        enc = self.tokenizer(
            str(row['caption']),
            padding='max_length', max_length=64,
            truncation=True, return_tensors='pt'
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        # scene 라벨
        scene_label = self.scene2idx.get(str(row['scene']), 0)

        return pixel_values, input_ids, attention_mask, scene_label

# --- 데이터 로드 및 분할 ---
df = pd.read_excel(CSV_PATH)

# [중요] 엑셀의 image_id가 실제 파일과 매칭되는지 전수 검사 (미리 에러 방지)
df['is_file_exist'] = df['image_id'].apply(lambda x: dn(x) in actual_file_map)
missing_files = df[df['is_file_exist'] == False]

if len(missing_files) > 0:
    print(f"⚠️ 경고: 엑셀 명단 중 {len(missing_files)}개의 파일이 폴더에 없습니다!")
    print(f"예시: {missing_files['image_id'].iloc[0]}")
    # 파일이 없는 행은 학습에서 아예 제외
    df = df[df['is_file_exist'] == True].copy()

print('\nscene 분포:')
print(df['scene'].value_counts())

unknown = set(df['scene'].unique()) - set(SCENE_CLASSES)
if unknown:
    print(f'\n⚠️ SCENE_CLASSES에 없는 값: {unknown}')

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(f'\n최종 Train: {len(train_df)}장 / Val: {len(val_df)}장')

# 데이터로더 생성
train_dataset = TravelDataset(train_df, IMAGE_ROOT, processor, text_tokenizer, SCENE2IDX)
val_dataset   = TravelDataset(val_df,   IMAGE_ROOT, processor, text_tokenizer, SCENE2IDX)

# num_workers=2가 가끔 코랩에서 에러를 내면 0으로 바꾸세요.
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train 배치 수: {len(train_loader)} / Val 배치 수: {len(val_loader)}')

🔎 이미지 폴더 스캔 중...
✅ 총 4846개의 실제 파일 인식 완료.
⚠️ 경고: 엑셀 명단 중 57개의 파일이 폴더에 없습니다!
예시: 숨도_5_공공3유형.jpg

scene 분포:
scene
전통     880
공원     819
바다     555
거리     333
숲      282
도시     278
산      242
전망     227
호수     214
조형물    213
카페     139
전시      18
온천       3
Name: count, dtype: int64

최종 Train: 3782장 / Val: 421장
Train 배치 수: 119 / Val 배치 수: 14


In [ ]:
import os
import time

# ==========================================
# 1. 학습 설정
# ==========================================
EPOCHS = 5  # LoRA는 이미 똑똑한 모델을 미세조정하는 거라 3~5 Epoch면 충분합니다.
SAVE_DIR = '/content/drive/MyDrive/checkpoints/stage2' # 저장할 폴더 경로
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"🏃‍♂️ Stage 2 학습을 시작합니다! (총 {EPOCHS} Epochs)")

# ==========================================
# 2. 학습 루프 (Training Loop)
# ==========================================
for epoch in range(EPOCHS):
    start_time = time.time()

    # 모델들을 학습 모드로 전환 (LoRA 어댑터와 Head들만 학습됨)
    text_encoder.train()
    vision_encoder.train()
    txt_proj_layer.train()
    proj_head.train()
    scene_head.train()

    epoch_loss = 0.0

    # train_loader는 Stage 1에서 쓰시던 것을 그대로 쓰시면 됩니다!
    for batch_idx, batch in enumerate(train_loader):
        # 데이터 장치(GPU)로 이동
        pixel_values, input_ids, attention_mask, scene_label = [b.to(device) for b in batch]

        # 앞서 정의한 train_step 함수 호출 (로스 계산 및 역전파)
        loss = train_step(pixel_values, input_ids, attention_mask, scene_label)
        epoch_loss += loss

        # 10배치마다 로그 출력
        if (batch_idx + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss:.4f}")

    avg_loss = epoch_loss / len(train_loader)
    elapsed = time.time() - start_time
    print(f"✅ Epoch {epoch+1} 완료! | 평균 Loss: {avg_loss:.4f} | 소요 시간: {elapsed:.0f}초\n")

# ==========================================
# 3. 모델 저장 (매우 중요 ⭐️)
# ==========================================
print("💾 학습 완료! 가벼운 LoRA 가중치와 Head만 분리해서 저장합니다...")

# 1) LoRA 가중치 저장 (PEFT 라이브러리 자체 기능 사용)
# 원본 모델(1.5GB)은 놔두고, 우리가 학습시킨 LoRA 어댑터(수 MB)만 저장합니다.
text_encoder.save_pretrained(os.path.join(SAVE_DIR, "text_lora"))
vision_encoder.save_pretrained(os.path.join(SAVE_DIR, "vision_lora"))

# 2) Custom Head들 저장
# 우리가 Stage 1부터 애지중지 키워온 Head 3개도 딕셔너리로 묶어서 저장합니다.
heads_state = {
    'txt_proj_layer': txt_proj_layer.state_dict(),
    'proj_head': proj_head.state_dict(),
    'scene_head': scene_head.state_dict()
}
torch.save(heads_state, os.path.join(SAVE_DIR, "custom_heads.pt"))

print(f"🎉 성공적으로 저장되었습니다! 저장 위치: {SAVE_DIR}")

🏃‍♂️ Stage 2 학습을 시작합니다! (총 5 Epochs)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/5] | Batch [10/119] | Loss: 4.0310
Epoch [1/5] | Batch [20/119] | Loss: 4.0837
Epoch [1/5] | Batch [30/119] | Loss: 4.3536
Epoch [1/5] | Batch [40/119] | Loss: 4.3041
Epoch [1/5] | Batch [50/119] | Loss: 4.1044
Epoch [1/5] | Batch [60/119] | Loss: 4.0422
Epoch [1/5] | Batch [70/119] | Loss: 4.1895
Epoch [1/5] | Batch [80/119] | Loss: 4.3717
Epoch [1/5] | Batch [90/119] | Loss: 4.0889
Epoch [1/5] | Batch [100/119] | Loss: 4.1405
Epoch [1/5] | Batch [110/119] | Loss: 3.8633
✅ Epoch 1 완료! | 평균 Loss: 4.0757 | 소요 시간: 2886초

Epoch [2/5] | Batch [10/119] | Loss: 3.9156
Epoch [2/5] | Batch [20/119] | Loss: 4.4302
Epoch [2/5] | Batch [30/119] | Loss: 3.6433
Epoch [2/5] | Batch [40/119] | Loss: 4.0330
Epoch [2/5] | Batch [50/119] | Loss: 4.1647
Epoch [2/5] | Batch [60/119] | Loss: 3.6978
Epoch [2/5] | Batch [70/119] | Loss: 3.7468
Epoch [2/5] | Batch [80/119] | Loss: 4.1492
Epoch [2/5] | Batch [90/119] | Loss: 3.5617
Epoch [2/5] | Batch [100/119] | Loss: 3.7056
Epoch [2/5] | Batch [110/

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 성공적으로 저장되었습니다! 저장 위치: /content/drive/MyDrive/checkpoints/stage2


In [ ]:
from google.colab import files

files.downloads(os.path.join(SAVE_DIR, "custom_heads.pt"))

AttributeError: module 'google.colab.files' has no attribute 'downloads'

In [ ]:
from google.colab import files

files.downloads(os.path.join(SAVE_DIR, "custom_heads.pt"))